# **Sentiment Analysis using NLP Pipeline & ML Models**


# 1.Data Understanding

In [5]:
import pandas as pd

url = "https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/IMDB-Dataset.csv"

df = pd.read_csv(url)

print(df.shape)
print(df['sentiment'].value_counts())
df.head()

(50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# 2. NLP Preprocessing

In [6]:
import re, nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stop = set(stopwords.words('english'))
stem = PorterStemmer()

def preprocess(text):
    text = text.lower()                              # lowercasing
    text = re.sub(r'http\S+|www\S+', '', text)        # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)              # remove punctuation/special chars
    words = text.split()                             # tokenization
    words = [w for w in words if w not in stop]       # remove stopwords
    words = [stem.stem(w) for w in words]             # stemming
    return " ".join(words)

df['clean_text'] = df['review'].apply(preprocess)

df[['review','clean_text']].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,review,clean_text
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod youll hook ...
1,A wonderful little production. <br /><br />The...,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic there famili littl boy jake think there ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


# 3. Feature Engineering

In [7]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Bag of Words
bow = CountVectorizer()
X_bow = bow.fit_transform(df['clean_text'])

# TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df['clean_text'])

X_bow.shape, X_tfidf.shape

((50000, 137381), (50000, 137381))

**Word2Vec**

In [9]:
!pip install gensim
from gensim.models import Word2Vec

tokens = [text.split() for text in df['clean_text']]

w2v = Word2Vec(tokens, vector_size=100, window=5, min_count=2)

w2v.wv['good']  # example vector

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 59.3 MB/s eta 0:00:00


array([ 1.7941358 , -0.24089654,  0.16462842,  0.41676906,  1.2800206 ,
       -0.12694442, -0.32682872,  0.7401808 , -0.29295143,  0.47263396,
       -0.83976823,  1.1571592 , -0.6392252 ,  0.8377371 , -1.8024621 ,
        1.2568058 ,  0.00889603,  1.938162  ,  0.22074531,  0.95088226,
       -0.06544221, -0.81688094,  1.1036563 ,  0.93448895,  0.53532547,
        0.4627219 , -1.9116064 ,  1.3432932 ,  0.13077097,  0.30201086,
        0.8533549 ,  1.5109949 , -2.3844895 , -0.5668619 ,  0.29266727,
       -1.5706261 ,  0.92192346,  0.82559794, -0.9972215 , -0.71766627,
        0.25187793, -0.01463559,  2.7523818 , -1.0572242 , -1.9961004 ,
        1.9827038 , -2.9182246 ,  0.7657018 ,  0.8780653 , -0.3173305 ,
        2.535174  ,  0.5106143 ,  0.37251666,  0.1278196 ,  0.6402949 ,
       -2.7021646 , -0.87432826, -1.7561911 , -2.3570564 , -1.7555133 ,
        0.2830333 ,  0.17332715, -0.04286093,  0.14024079,  0.369331  ,
        0.77135015,  0.36014062, -0.95300764, -1.710728  ,  0.13

**Avg Word2Vec**

In [10]:
import numpy as np

def avg_w2v(words, model):
    vec = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vec, axis=0) if len(vec)>0 else np.zeros(100)

X_w2v = np.array([avg_w2v(text.split(), w2v) for text in df['clean_text']])

X_w2v.shape

(50000, 100)

# 4. Model Building

**Train-Test Split**

In [11]:
from sklearn.model_selection import train_test_split

X = X_tfidf   # you can also try X_bow
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

**Logistic Regression**

In [12]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

LogisticRegression()

**Naive Bayes**

In [13]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

MultinomialNB()

**Decision Tree**

In [14]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

DecisionTreeClassifier()

**Random Forest**

In [15]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)

RandomForestClassifier()

# 5. Model Evaluation

In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, name):
    pred = model.predict(X_test)
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred, average='weighted'))
    print("Recall:", recall_score(y_test, pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, pred, average='weighted'))

# Evaluate models
evaluate(lr, "Logistic Regression")
evaluate(nb, "Naive Bayes")
evaluate(dt, "Decision Tree")


Logistic Regression
Accuracy: 0.889
Precision: 0.8892646769145119
Recall: 0.889
F1 Score: 0.889

Naive Bayes
Accuracy: 0.8622
Precision: 0.862291779026209
Recall: 0.8622
F1 Score: 0.8621728575058234

Decision Tree
Accuracy: 0.7226
Precision: 0.7226112221595901
Recall: 0.7226
F1 Score: 0.7226041947772788


**Compare in Table**

In [18]:
import pandas as pd

models = {"LR": lr, "NB": nb, "DT": dt}
results = []

for name, model in models.items():
    pred = model.predict(X_test)
    results.append([
        name,
        accuracy_score(y_test, pred),
        precision_score(y_test, pred, average='weighted'),
        recall_score(y_test, pred, average='weighted'),
        f1_score(y_test, pred, average='weighted')
    ])

pd.DataFrame(results, columns=["Model","Accuracy","Precision","Recall","F1"])

,Model,Accuracy,Precision,Recall,F1
0,LR,0.8890,0.889265,0.8890,0.889000
1,NB,0.8622,0.862292,0.8622,0.862173
2,DT,0.7226,0.722611,0.7226,0.722604


# 6. Comparison & Insights


TF-IDF performed better than BoW because it gives importance to important words.

Preprocessing improved model accuracy by removing noise.

Logistic Regression gave the best performance.

Naive Bayes was fast but slightly less accurate.

Decision Tree may overfit on text data.


**Trade-offs:**
BoW → simple but less accurate
TF-IDF → better performance
Logistic Regression → best balance of accuracy and speed

# **Final Pipeline**

Raw Data → Preprocessing → Feature Engineering → Model → Evaluation